# Introduction

<center><img src="https://i.imgur.com/9hLRsjZ.jpg" height=400></center>

This dataset was scraped from [nextspaceflight.com](https://nextspaceflight.com/launches/past/?page=1) and includes all the space missions since the beginning of Space Race between the USA and the Soviet Union in 1957!

### Install Package with Country Codes

In [ ]:
%pip install iso3166

### Upgrade Plotly

Run the cell below if you are working with Google Colab.

In [ ]:
%pip install --upgrade plotly

### Import Statements

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
# These might be helpful:
from iso3166 import countries
from datetime import datetime, timedelta

### Notebook Presentation

In [2]:
pd.options.display.float_format = '{:,.2f}'.format

### Load the Data

In [3]:
df_data = pd.read_csv('mission_launches.csv')

# Preliminary Data Exploration

* What is the shape of `df_data`? 
* How many rows and columns does it have?
* What are the column names?
* Are there any NaN values or duplicates?

In [4]:
df_data.head()

,Unnamed: 0.1,Unnamed: 0,Organisation,Location,Date,Detail,Rocket_Status,Price,Mission_Status
0,0,0,SpaceX,"LC-39A, Kennedy Space Center, Florida, USA","Fri Aug 07, 2020 05:12 UTC",Falcon 9 Block 5 | Starlink V1 L9 & BlackSky,StatusActive,50.0,Success
1,1,1,CASC,"Site 9401 (SLS-2), Jiuquan Satellite Launch Ce...","Thu Aug 06, 2020 04:01 UTC",Long March 2D | Gaofen-9 04 & Q-SAT,StatusActive,29.75,Success
2,2,2,SpaceX,"Pad A, Boca Chica, Texas, USA","Tue Aug 04, 2020 23:57 UTC",Starship Prototype | 150 Meter Hop,StatusActive,NaN,Success
3,3,3,Roscosmos,"Site 200/39, Baikonur Cosmodrome, Kazakhstan","Thu Jul 30, 2020 21:25 UTC",Proton-M/Briz-M | Ekspress-80 & Ekspress-103,StatusActive,65.0,Success
4,4,4,ULA,"SLC-41, Cape Canaveral AFS, Florida, USA","Thu Jul 30, 2020 11:50 UTC",Atlas V 541 | Perseverance,StatusActive,145.0,Success


In [5]:
df_data.shape

(4324, 9)

In [6]:
df_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 4324 entries, 0 to 4323
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Unnamed: 0.1    4324 non-null   int64
 1   Unnamed: 0      4324 non-null   int64
 2   Organisation    4324 non-null   str  
 3   Location        4324 non-null   str  
 4   Date            4324 non-null   str  
 5   Detail          4324 non-null   str  
 6   Rocket_Status   4324 non-null   str  
 7   Price           964 non-null    str  
 8   Mission_Status  4324 non-null   str  
dtypes: int64(2), str(7)
memory usage: 304.2 KB


In [7]:
df_data.describe()

,Unnamed: 0.1,Unnamed: 0
count,"4,324.00","4,324.00"
mean,"2,161.50","2,161.50"
std,"1,248.38","1,248.38"
min,0.00,0.00
25%,"1,080.75","1,080.75"
50%,"2,161.50","2,161.50"
75%,"3,242.25","3,242.25"
max,"4,323.00","4,323.00"


In [8]:
df_data.isna().sum()

Unnamed: 0.1         0
Unnamed: 0           0
Organisation         0
Location             0
Date                 0
Detail               0
Rocket_Status        0
Price             3360
Mission_Status       0
dtype: int64

In [9]:
percent_missing = df_data.isna().sum() / len(df_data) * 100
print(percent_missing)

Unnamed: 0.1      0.00
Unnamed: 0        0.00
Organisation      0.00
Location          0.00
Date              0.00
Detail            0.00
Rocket_Status     0.00
Price            77.71
Mission_Status    0.00
dtype: float64


In [10]:
df_data.duplicated().sum()

np.int64(0)

In [11]:
print(f"Null values for each column:\n{df_data.isnull().sum()}\n")
print(f"Percentage of null values for each column:\n{df_data.isnull().mean() * 100}")


Null values for each column:
Unnamed: 0.1         0
Unnamed: 0           0
Organisation         0
Location             0
Date                 0
Detail               0
Rocket_Status        0
Price             3360
Mission_Status       0
dtype: int64

Percentage of null values for each column:
Unnamed: 0.1      0.00
Unnamed: 0        0.00
Organisation      0.00
Location          0.00
Date              0.00
Detail            0.00
Rocket_Status     0.00
Price            77.71
Mission_Status    0.00
dtype: float64


So, shape is (4324, 9), we have 9 columns and 4324 rows.
The 2 initial columns don't make any sense, so they will be dropped
Plus, for the column price only 22% of price is present, so we need to fill the Nan values to avoid losing relevant data. 
There are no duplicates. Good. 

## Data Cleaning - Check for Missing Values and Duplicates

Consider removing columns containing junk data. 

In [12]:
clean_df = df_data.drop(columns=['Unnamed: 0.1', 'Unnamed: 0'])

In [13]:
clean_df.head()

,Organisation,Location,Date,Detail,Rocket_Status,Price,Mission_Status
0,SpaceX,"LC-39A, Kennedy Space Center, Florida, USA","Fri Aug 07, 2020 05:12 UTC",Falcon 9 Block 5 | Starlink V1 L9 & BlackSky,StatusActive,50.0,Success
1,CASC,"Site 9401 (SLS-2), Jiuquan Satellite Launch Ce...","Thu Aug 06, 2020 04:01 UTC",Long March 2D | Gaofen-9 04 & Q-SAT,StatusActive,29.75,Success
2,SpaceX,"Pad A, Boca Chica, Texas, USA","Tue Aug 04, 2020 23:57 UTC",Starship Prototype | 150 Meter Hop,StatusActive,NaN,Success
3,Roscosmos,"Site 200/39, Baikonur Cosmodrome, Kazakhstan","Thu Jul 30, 2020 21:25 UTC",Proton-M/Briz-M | Ekspress-80 & Ekspress-103,StatusActive,65.0,Success
4,ULA,"SLC-41, Cape Canaveral AFS, Florida, USA","Thu Jul 30, 2020 11:50 UTC",Atlas V 541 | Perseverance,StatusActive,145.0,Success


In [14]:
clean_df['Price'] = pd.to_numeric(clean_df['Price'], errors='coerce')


We decide not to fillna neither neither dropna. There is a reason why a mission doesn't have the price. This information is unknown. 
On the one hand, if we use fillna with 0, we will provide a misleading information stating that the mission had no costs.
On the other hand, if we drop columns we lose valuable information that will be precious into the data analysis and to extract precious insights.  

## Descriptive Statistics

In [15]:

print(f"Value counts for Organisation:\n{clean_df['Organisation'].value_counts().head(10)}\n")
print(f"Relative frequencies for Organisation:\n{clean_df['Organisation'].value_counts(normalize=True).head(10)}\n")



Value counts for Organisation:
Organisation
RVSN USSR           1777
Arianespace          279
CASC                 251
General Dynamics     251
NASA                 203
VKS RF               201
US Air Force         161
ULA                  140
Boeing               136
Martin Marietta      114
Name: count, dtype: int64

Relative frequencies for Organisation:
Organisation
RVSN USSR          0.41
Arianespace        0.06
CASC               0.06
General Dynamics   0.06
NASA               0.05
VKS RF             0.05
US Air Force       0.04
ULA                0.03
Boeing             0.03
Martin Marietta    0.03
Name: proportion, dtype: float64



In [16]:
print(f"Distinct values for Organisation:\n{clean_df['Organisation'].unique()[:10].tolist()}\n")
print(f"Distinct values for Mission_Status:\n{clean_df['Mission_Status'].unique()[:10].tolist()}\n")
print(f"Number of distinct values for Mission_Status:\n{clean_df['Mission_Status'].nunique()}\n")
print(f"Value counts for Mission_Status:\n{clean_df['Mission_Status'].value_counts()}\n")

Distinct values for Organisation:
['SpaceX', 'CASC', 'Roscosmos', 'ULA', 'JAXA', 'Northrop', 'ExPace', 'IAI', 'Rocket Lab', 'Virgin Orbit']

Distinct values for Mission_Status:
['Success', 'Failure', 'Prelaunch Failure', 'Partial Failure']

Number of distinct values for Mission_Status:
4

Value counts for Mission_Status:
Mission_Status
Success              3879
Failure               339
Partial Failure       102
Prelaunch Failure       4
Name: count, dtype: int64



In [17]:
print(f"Descriptive statistics for Price:\n{clean_df['Price'].describe()}\n")
print(f"Median for Price:\n{clean_df['Price'].median()}\n")
print(f"Skewness for Price:\n{clean_df['Price'].skew()}\n")
print(f"Missions with known price: {clean_df['Price'].count()} / {len(clean_df)}")

Descriptive statistics for Price:
count   949.00
mean    129.80
std     143.22
min       5.30
25%      40.00
50%      62.00
75%     164.00
max     450.00
Name: Price, dtype: float64

Median for Price:
62.0

Skewness for Price:
1.5077937711635496

Missions with known price: 949 / 4324


In [18]:
print(f"Value counts for Rocket_Status:\n{clean_df['Rocket_Status'].value_counts()}\n")
print(f"Relative frequencies for Rocket_Status:\n{clean_df['Rocket_Status'].value_counts(normalize=True)}\n")

Value counts for Rocket_Status:
Rocket_Status
StatusRetired    3534
StatusActive      790
Name: count, dtype: int64

Relative frequencies for Rocket_Status:
Rocket_Status
StatusRetired   0.82
StatusActive    0.18
Name: proportion, dtype: float64



In [19]:
print(f"Value counts for Mission_Status:\n{clean_df['Mission_Status'].value_counts()}\n")
print(f"Relative frequencies for Mission_Status:\n{clean_df['Mission_Status'].value_counts(normalize=True)}\n")

Value counts for Mission_Status:
Mission_Status
Success              3879
Failure               339
Partial Failure       102
Prelaunch Failure       4
Name: count, dtype: int64

Relative frequencies for Mission_Status:
Mission_Status
Success             0.90
Failure             0.08
Partial Failure     0.02
Prelaunch Failure   0.00
Name: proportion, dtype: float64



# Number of Launches per Company

Create a chart that shows the number of space mission launches by organisation.

In [20]:
top_org_launches = clean_df['Organisation'].value_counts().head(15).reset_index()
top_org_launches.columns = ['Organisation', 'Launches']

In [21]:
fig = px.bar(
    top_org_launches,
    x='Organisation',
    y='Launches',
    title='Top 10 Organisations by Number of Space Mission Launches',
    color='Launches',
    color_continuous_scale='Agsunset'
)
fig.update_layout(
    xaxis_title='Organisation',
    yaxis_title='Number of Launches',
    coloraxis_showscale=False)
fig.show()



# Number of Active versus Retired Rockets

How many rockets are active compared to those that are decomissioned? 

In [22]:
rocket_status_map = {'StatusActive': 'Active', 'StatusRetired': 'Retired'}
top_org_act_rockets = clean_df['Rocket_Status'].replace(rocket_status_map).value_counts().reset_index()
top_org_act_rockets.columns = ['Rocket Status', 'Count']

In [25]:
fig = px.bar(
    top_org_act_rockets,
    x='Rocket Status',
    y='Count',
    title='Active vs Retired Rockets',
    color='Rocket Status',
    color_discrete_map={'Active': '#2ecc71', 'Retired': '#e74c3c'},
    text='Count'
)
fig.update_layout(
    xaxis_title='Rocket Status',
    yaxis_title='Number of Missions',
    showlegend=False
)
fig.update_traces(textposition='outside')
fig.show()

# Distribution of Mission Status

How many missions were successful?
How many missions failed?

In [23]:
mission_status_df = clean_df['Mission_Status'].value_counts().reset_index()
mission_status_df.columns = ['Mission Status', 'Count']
mission_status_df.sort_values('Count', ascending=True, inplace=True)


In [24]:
fig = px.bar(
    mission_status_df,
    x='Count',
    y='Mission Status',
    orientation='h',
    title='Distribution of Mission Status',
    color='Count',
    color_continuous_scale='RdYlGn',
    text='Count'
)
fig.update_layout(
    xaxis_title='Number of Missions',
    yaxis_title='Mission Status',
    coloraxis_showscale=False,
    margin=dict(r=80)
)
fig.update_traces(textposition='outside', cliponaxis=False)
fig.show()


# How Expensive are the Launches? 

Create a histogram and visualise the distribution. The price column is given in USD millions (careful of missing values). 

In [25]:
fig = px.histogram(
    clean_df.dropna(subset=['Price']),
    x='Price',
    nbins=40,
    title='Distribution of Launch Prices (USD Millions)',
    color_discrete_sequence=['steelblue']
)
fig.update_layout(
    xaxis=dict(
        title='Price (USD Millions)',
        dtick=50,          # main tick every 50
        tick0=0,
        minor=dict(
            dtick=25,      # intermediate tick every 25
            ticks='outside',
            ticklen=4,
            showgrid=False
        )
    ),
    yaxis_title='Number of Missions',
    bargap=0.05
)
fig.show()


# Use a Choropleth Map to Show the Number of Launches by Country

* Create a choropleth map using [the plotly documentation](https://plotly.com/python/choropleth-maps/)
* Experiment with [plotly's available colours](https://plotly.com/python/builtin-colorscales/). I quite like the sequential colour `matter` on this map. 
* You'll need to extract a `country` feature as well as change the country names that no longer exist.

Wrangle the Country Names

You'll need to use a 3 letter country code for each country. You might have to change some country names.

* Russia is the Russian Federation
* New Mexico should be USA
* Yellow Sea refers to China
* Shahrud Missile Test Site should be Iran
* Pacific Missile Range Facility should be USA
* Barents Sea should be Russian Federation
* Gran Canaria should be USA


You can use the iso3166 package to convert the country names to Alpha3 format.

In [26]:
clean_df['Country'] = clean_df['Location'].str.split(',').str[-1].str.strip()
country_fixes = {
    'Russia':                       'Russian Federation',
    'New Mexico':                   'USA',
    'Yellow Sea':                   'China',
    'Shahrud Missile Test Site':    'Iran, Islamic Republic of',
    'Pacific Missile Range Facility': 'USA',
    'Barents Sea':                  'Russian Federation',
    'Gran Canaria':                 'Spain',
    'Pacific Ocean':                'USA',
    'Iran':                         'Iran, Islamic Republic of',
    'South Korea':                  'Korea, Republic of',
    'North Korea':                  'Korea, Democratic People\'s Republic of',
    'United States':                'USA',
}
clean_df['Country'] = clean_df['Country'].replace(country_fixes)


In [27]:
clean_df.head()

,Organisation,Location,Date,Detail,Rocket_Status,Price,Mission_Status,Country
0,SpaceX,"LC-39A, Kennedy Space Center, Florida, USA","Fri Aug 07, 2020 05:12 UTC",Falcon 9 Block 5 | Starlink V1 L9 & BlackSky,StatusActive,50.00,Success,USA
1,CASC,"Site 9401 (SLS-2), Jiuquan Satellite Launch Ce...","Thu Aug 06, 2020 04:01 UTC",Long March 2D | Gaofen-9 04 & Q-SAT,StatusActive,29.75,Success,China
2,SpaceX,"Pad A, Boca Chica, Texas, USA","Tue Aug 04, 2020 23:57 UTC",Starship Prototype | 150 Meter Hop,StatusActive,NaN,Success,USA
3,Roscosmos,"Site 200/39, Baikonur Cosmodrome, Kazakhstan","Thu Jul 30, 2020 21:25 UTC",Proton-M/Briz-M | Ekspress-80 & Ekspress-103,StatusActive,65.00,Success,Kazakhstan
4,ULA,"SLC-41, Cape Canaveral AFS, Florida, USA","Thu Jul 30, 2020 11:50 UTC",Atlas V 541 | Perseverance,StatusActive,145.00,Success,USA


In [28]:
def to_iso3(name):
    try:
        return countries.get(name).alpha3
    except (KeyError, AttributeError):
        return None

clean_df['ISO3'] = clean_df['Country'].apply(to_iso3)

# Diagnostica: verifica se sono rimasti valori non mappati
unmapped = clean_df[clean_df['ISO3'].isna()]['Country'].unique()
print("Non mappati:", unmapped)

Non mappati: []


In [29]:
df_country_launches = clean_df['ISO3'].value_counts().reset_index()
df_country_launches.columns = ['Country', 'Launches']

In [33]:
df_country_launches.head()

,Country,Launches
0,RUS,1398
1,USA,1385
2,KAZ,701
3,FRA,303
4,CHN,269


In [30]:
fig = go.Figure(data=go.Choropleth(
    locations = df_country_launches['Country'],
    z = df_country_launches['Launches'],
    text = df_country_launches['Country'],
    locationmode = 'ISO-3',
    colorscale = 'matter',
    autocolorscale=False,
    reversescale=True,
    marker_line_width=0.5,
    colorbar_tickprefix = '',
    colorbar=dict(len=1, 
                  thickness=20, 
                  title=dict(text='Launches<br>by country',
                      font=dict(
                          size=10)
                          )
                  ),
))

fig.update_layout(
    title_text='Number of Launches by Country',
    geo=dict(
        showframe=False,
        showcoastlines=False,
        projection_type='equirectangular'
    )
)

fig.show()

# Use a Choropleth Map to Show the Number of Failures by Country


In [31]:
df_country_failures = clean_df[clean_df['Mission_Status'] == 'Failure']['ISO3'].value_counts().reset_index()
df_country_failures.columns = ['Country', 'Failures']

In [32]:
df_country_failures.head()

,Country,Failures
0,USA,132
1,KAZ,72
2,RUS,63
3,CHN,19
4,FRA,13


In [33]:
fig = go.Figure(data=go.Choropleth(
    locations = df_country_failures['Country'],
    z = df_country_failures['Failures'],
    text = df_country_failures['Country'],
    locationmode = 'ISO-3',
    colorscale = 'matter',
    autocolorscale=False,
    reversescale=True,
    marker_line_width=0.5,
    colorbar_tickprefix = '',
    colorbar=dict(len=1, 
                  thickness=20, 
                  title=dict(text='Failures<br>by country',
                      font=dict(
                          size=10)
                          )
                  ),
))

fig.update_layout(
    title_text='Number of Failures by Country',
    geo=dict(
        showframe=False,
        showcoastlines=False,
        projection_type='equirectangular'
    )
)

fig.show()

# Create a Plotly Sunburst Chart of the countries, organisations, and mission status. 

In [34]:
df_sunburst_org = clean_df.groupby(['Country','Organisation','Mission_Status']).size().reset_index(name='Missions')
df_sunburst_org.columns = ['country','organisation','Mission_Status','Missions']
df_sunburst_org.sort_values('Missions', ascending=False ,inplace=True)
df_sunburst_org.head()

,country,organisation,Mission_Status,Missions
81,Russian Federation,RVSN USSR,Success,1119
58,Kazakhstan,RVSN USSR,Success,495
19,France,Arianespace,Success,267
9,China,CASC,Success,231
104,USA,General Dynamics,Success,203


In [35]:
fig = px.sunburst(df_sunburst_org, 
                  path=['country', 'organisation','Mission_Status'], 
                  values='Missions',
                  color='Missions', 
                  title='Where do Discoveries Take Place?')
fig.update_layout(xaxis_title='Number of Missions',
                  yaxis_title='Country',
                  coloraxis_showscale=False )
fig.show()

# Analyse the Total Amount of Money Spent by Organisation on Space Missions

In [36]:
df_money_by_org = clean_df[clean_df['Price'].notnull()].groupby('Organisation')['Price'].sum().reset_index()
df_money_by_org.sort_values('Price', ascending=False, inplace=True)
df_money_by_org = df_money_by_org.head(15).sort_values('Price', ascending=True)
df_money_by_org['Price'] = df_money_by_org['Price'].round(2)

In [37]:
fig = px.bar(
    df_money_by_org,
    x='Price',
    y='Organisation',
    orientation='h',
    title='Total Amount of Money Spent by Organisation on Space Missions',
    color='Price',
    color_continuous_scale='Plasma',
    text='Price'
)
fig.update_layout(
    xaxis_title='Money Spent',
    yaxis_title='Organisation name',
    coloraxis_showscale=False,
    margin=dict(r=80)
)
fig.update_traces(textposition='outside', cliponaxis=False)
fig.show()

# Analyse the Amount of Money Spent by Organisation per Launch

In [38]:
df_avg_price_by_launch = clean_df[clean_df['Price'].notnull()].groupby('Organisation')['Price'].mean().reset_index()
df_avg_price_by_launch.sort_values('Price', ascending=False, inplace=True)
#df_avg_price_by_launch.head()
df_avg_price_by_launch = df_avg_price_by_launch.head(15).sort_values('Price', ascending=True)
df_avg_price_by_launch['Price'] = df_avg_price_by_launch['Price'].round(2)

In [39]:
fig = px.bar(
    df_avg_price_by_launch,
    x='Price',
    y='Organisation',
    orientation='h',
    title='Average Amount of Money Spent by Organisation on Space Missions',
    color='Price',
    color_continuous_scale='Plasma',
    text='Price'
)
fig.update_layout(
    xaxis_title='Money Spent',
    yaxis_title='Organisation name',
    coloraxis_showscale=False,
    margin=dict(r=80)
)
fig.update_traces(textposition='outside', cliponaxis=False)
fig.show()

# Chart the Number of Launches per Year

In [40]:
clean_df['Year_of_Launch'] = pd.to_datetime(clean_df['Date'], errors='coerce').dt.year
yearly_launches = clean_df.groupby('Year_of_Launch').size().reset_index(name='Launches')


In [41]:
fig = px.line(yearly_launches, 
              x='Year_of_Launch', 
              y='Launches', title='Number of Space Mission Launches per Year')
fig.update_layout(
    xaxis=dict(
        title='Year of Launch',
        dtick=10,          # main tick every 10 years
        tick0=1950,
        minor=dict(
            dtick=5,       # intermediate tick every 5 years
            ticks='outside',
            ticklen=4,
            showgrid=False
        )
    ),
    yaxis_title='Number of Launches'
)
fig.show()

# Chart the Number of Launches Month-on-Month until the Present

Which month has seen the highest number of launches in all time? Superimpose a rolling average on the month on month time series chart. 

In [42]:
clean_df['RealDate'] = pd.to_datetime(clean_df['Date'], format='mixed', utc=True, errors='coerce')
print(clean_df['RealDate'].head())

0   2020-08-07 05:12:00+00:00
1   2020-08-06 04:01:00+00:00
2   2020-08-04 23:57:00+00:00
3   2020-07-30 21:25:00+00:00
4   2020-07-30 11:50:00+00:00
Name: RealDate, dtype: datetime64[us, UTC]


In [43]:
clean_df['Month_of_Launch'] = clean_df['RealDate'].dt.to_period('M')
clean_df.head()

C:\Users\ilpot\AppData\Local\Temp\ipykernel_21004\1604958573.py:1: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  clean_df['Month_of_Launch'] = clean_df['RealDate'].dt.to_period('M')


,Organisation,Location,Date,Detail,Rocket_Status,Price,Mission_Status,Country,ISO3,Year_of_Launch,RealDate,Month_of_Launch
0,SpaceX,"LC-39A, Kennedy Space Center, Florida, USA","Fri Aug 07, 2020 05:12 UTC",Falcon 9 Block 5 | Starlink V1 L9 & BlackSky,StatusActive,50.00,Success,USA,USA,"2,020.00",2020-08-07 05:12:00+00:00,2020-08
1,CASC,"Site 9401 (SLS-2), Jiuquan Satellite Launch Ce...","Thu Aug 06, 2020 04:01 UTC",Long March 2D | Gaofen-9 04 & Q-SAT,StatusActive,29.75,Success,China,CHN,"2,020.00",2020-08-06 04:01:00+00:00,2020-08
2,SpaceX,"Pad A, Boca Chica, Texas, USA","Tue Aug 04, 2020 23:57 UTC",Starship Prototype | 150 Meter Hop,StatusActive,NaN,Success,USA,USA,"2,020.00",2020-08-04 23:57:00+00:00,2020-08
3,Roscosmos,"Site 200/39, Baikonur Cosmodrome, Kazakhstan","Thu Jul 30, 2020 21:25 UTC",Proton-M/Briz-M | Ekspress-80 & Ekspress-103,StatusActive,65.00,Success,Kazakhstan,KAZ,"2,020.00",2020-07-30 21:25:00+00:00,2020-07
4,ULA,"SLC-41, Cape Canaveral AFS, Florida, USA","Thu Jul 30, 2020 11:50 UTC",Atlas V 541 | Perseverance,StatusActive,145.00,Success,USA,USA,"2,020.00",2020-07-30 11:50:00+00:00,2020-07


In [45]:
df_monthly_launches = clean_df.groupby('Month_of_Launch').size().reset_index(name='Launches')
df_monthly_launches['Rolling_Avg'] = df_monthly_launches['Launches'].rolling(window=12).mean()
df_monthly_launches.head()

,Month_of_Launch,Launches,Rolling_Avg
0,1957-10,1,NaN
1,1957-11,1,NaN
2,1957-12,1,NaN
3,1958-02,2,NaN
4,1958-03,3,NaN


In [46]:
plt.figure(figsize=(16,8), dpi=200) #dpi to make the line sharper
fig = px.line(df_monthly_launches, 
              x=df_monthly_launches['Month_of_Launch'].astype(str), 
              y='Rolling_Avg', title='Number of Space Mission Launches per Month (12-month rolling average)',
              color_discrete_sequence=['steelblue'] ,
              labels={'Month_of_Launch': 'Month of Launch', 'Rolling_Avg': '12-Month Rolling Average of Launches'}
              )
fig.update_layout(
                xaxis=dict(
                    title='Month of Launch',
                    nticks=20,          # main tick every 20 months
                ),
                yaxis_title='Number of Launches',
                coloraxis_showscale=False                         
                )
fig.show()

<Figure size 3200x1600 with 0 Axes>

# Launches per Month: Which months are most popular and least popular for launches?

Some months have better weather than others. Which time of year seems to be best for space missions?

In [47]:
df_month_of_year_launches = clean_df.groupby(clean_df['RealDate'].dt.month).size().reset_index(name='Launches')
df_month_of_year_launches['Month'] = pd.to_datetime(df_month_of_year_launches['RealDate'], format='%m').dt.month_name()
df_month_of_year_launches.head(12)

,RealDate,Launches,Month
0,1,268,January
1,2,336,February
2,3,353,March
3,4,383,April
4,5,326,May
5,6,402,June
6,7,351,July
7,8,373,August
8,9,365,September
9,10,381,October


In [48]:
fig = px.bar(df_month_of_year_launches, 
              x='Month', 
              y='Launches', title='Number of Space Mission Launches per Month (Yearly average)',
              color ='Launches',
              color_continuous_scale='Agsunset',
              labels={'Month_of_Launch': 'Month of Launch', 'Rolling_Avg': '12-Month Rolling Average of Launches'}
              )
fig.update_layout(
                xaxis=dict(
                    title='Month of Launch',
                    nticks=20,          # main tick every 20 months
                ),
                yaxis_title='Number of Launches',
                coloraxis_showscale=False                         
                )
fig.show()


# How has the Launch Price varied Over Time? 

Create a line chart that shows the average price of rocket launches over time. 

In [49]:
df_price_by_year = clean_df[clean_df['Price'].notnull()].groupby('Year_of_Launch')['Price'].mean().reset_index()
df_price_by_year.head()

,Year_of_Launch,Price
0,"1,964.00",63.23
1,"1,965.00",63.23
2,"1,966.00",59.00
3,"1,967.00",59.00
4,"1,968.00",59.00


In [52]:
fig = px.line(df_price_by_year, 
              x='Year_of_Launch', 
              y='Price', title='Average Price of Space Missions per Year (based on missions<br> with known price only)')
fig.update_layout(
    xaxis=dict(
        title='Year of Launch',
        dtick=10,          # main tick every 10 years
        tick0=1950,
        minor=dict(
            dtick=5,       # intermediate tick every 5 years
            ticks='outside',
            ticklen=4,
            showgrid=False
        )
    ),
    yaxis_title='Average Price'
)
fig.show()

# Chart the Number of Launches over Time by the Top 10 Organisations. 

How has the dominance of launches changed over time between the different players? 

In [72]:
top_10_orgs = clean_df['Organisation'].value_counts().head(10).reset_index()
top_10_orgs.columns = ['Organisation', 'Launches']
df_top_10_orgs = clean_df[clean_df['Organisation'].isin(top_10_orgs['Organisation'])]
df_top_10_orgs = df_top_10_orgs.groupby(['Year_of_Launch', 'Organisation']).size().reset_index(name='Launches')
df_top_10_orgs.head(10)

,Year_of_Launch,Organisation,Launches
0,"1,957.00",RVSN USSR,2
1,"1,958.00",NASA,2
2,"1,958.00",RVSN USSR,5
3,"1,958.00",US Air Force,2
4,"1,959.00",General Dynamics,1
5,"1,959.00",NASA,1
6,"1,959.00",RVSN USSR,4
7,"1,959.00",US Air Force,10
8,"1,960.00",General Dynamics,5
9,"1,960.00",NASA,4


In [80]:
fig = px.line(df_top_10_orgs, 
              x='Year_of_Launch', 
              y='Launches', title='Top 10 Organisations by Number of Space Mission Launches',
              color='Organisation'
              )
fig.update_layout(
        xaxis=dict(
        title='Year of Launch',
        dtick=10,          # main tick every 10 years
        tick0=1950,
        minor=dict(
            dtick=5,       # intermediate tick every 5 years
            ticks='outside',
            ticklen=4,
            showgrid=False
        )
    ),
    yaxis_title='Number of Launches',
    coloraxis_showscale=False)
fig.show()

# Cold War Space Race: USA vs USSR

The cold war lasted from the start of the dataset up until 1991. 

In [115]:
cold_war_df = clean_df[clean_df['Year_of_Launch'] <= 1991].copy()
cold_war_df['Country'].unique()
cold_war_df.shape

(2520, 12)

In [116]:
ussr_countries = ['Russian Federation', 'Kazakhstan', 'USSR']  # aggiungi altri se li trovi
usa_countries = ['USA', 'United States', 'United States of America']  # aggiungi altri se li trovi
cold_war_df['Superpower'] = None
cold_war_df.loc[cold_war_df['Country'].isin(ussr_countries), 'Superpower'] = 'USSR'
cold_war_df.loc[cold_war_df['Country'].isin(usa_countries), 'Superpower'] = 'USA'
cold_war_df = cold_war_df[cold_war_df['Superpower'].notnull()]
cold_war_df_pie = cold_war_df.groupby(['Superpower']).size().reset_index(name='Launches')


In [114]:
cold_war_df.shape

(2, 2)

## Create a Plotly Pie Chart comparing the total number of launches of the USSR and the USA

Hint: Remember to include former Soviet Republics like Kazakhstan when analysing the total number of launches. 

In [117]:
fig = px.pie(cold_war_df_pie,
             names='Superpower',
                values='Launches',
                title='Number of Space Mission Launches by Superpower during the Cold War',
                color_discrete_map={'USA': '#3498db', 'USSR': '#e74c3c'}
                )
fig.update_layout(
    title_text='Number of Space Mission Launches<br> by Superpower during the Cold War',
    geo=dict(
        showframe=False,
        showcoastlines=False,
        projection_type='equirectangular'
    )
)
fig.show()

## Create a Chart that Shows the Total Number of Launches Year-On-Year by the Two Superpowers

In [118]:
df_yby_superpowers_comparison = cold_war_df.groupby(['Year_of_Launch', 'Superpower']).size().reset_index(name='Launches')
df_yby_superpowers_comparison.head()

,Year_of_Launch,Superpower,Launches
0,"1,957.00",USA,1
1,"1,957.00",USSR,2
2,"1,958.00",USA,17
3,"1,958.00",USSR,5
4,"1,959.00",USA,16


In [113]:
print(cold_war_df.shape)

(2, 2)


In [121]:
fig = px.line(df_yby_superpowers_comparison, 
              x='Year_of_Launch', 
              y='Launches', title='Number of Space Mission Launches<br> per Year by Superpower during the Cold War',
              color='Superpower',
              color_discrete_map={'USA': '#3498db', 'USSR': '#e74c3c'}
              )
fig.update_layout(
    xaxis=dict(
        title='Year of Launch',
        dtick=5,          # main tick every 5 years
        tick0=1950,
        minor=dict(
            dtick=1,       # intermediate tick every 1 year
            ticks='outside',
            ticklen=4,
            showgrid=False
        )
    ),
    yaxis_title='Number of Launches',
    coloraxis_showscale=False
)
fig.show()

## Chart the Total Number of Mission Failures Year on Year.

In [129]:
clean_df['Mission_Status'].unique()


<StringArray>
['Success', 'Failure', 'Prelaunch Failure', 'Partial Failure']
Length: 4, dtype: str

In [135]:
df_monthly_failures = clean_df[clean_df['Mission_Status'].isin(['Failure', 'Prelaunch Failure', 'Partial Failure'])].groupby('Year_of_Launch').size().reset_index(name='Failures')
df_monthly_failures.head()

,Year_of_Launch,Failures
0,"1,957.00",1
1,"1,958.00",16
2,"1,959.00",12
3,"1,960.00",19
4,"1,961.00",20


In [150]:
fig = px.line(df_monthly_failures,
              x='Year_of_Launch',
              y='Failures', 
              title='Number of Space Mission Failures per Year during the Cold War',
              color_discrete_sequence=['steelblue'] ,
              labels={'Month_of_Launch': 'Month of Launch', 'Failures': 'Number of Failures'}
                )
fig.update_layout(
                xaxis=dict(
                    title='Month of Launch',
                    nticks=20,          # main tick every 20 months
                ),
                yaxis_title='Number of Failures',
                coloraxis_showscale=False                         
                )
fig.show()

## Chart the Percentage of Failures over Time

Did failures go up or down over time? Did the countries get better at minimising risk and improving their chances of success over time? 

In [156]:
df_monthly_failure_perc = clean_df[clean_df['Mission_Status'].isin(['Failure', 'Prelaunch Failure', 'Partial Failure'])].groupby('Year_of_Launch').size() / clean_df.groupby('Year_of_Launch').size()*100
df_monthly_failure_perc = df_monthly_failure_perc.reset_index(name='Failure_Rate')
df_monthly_failure_perc['Success_Rate'] = 100 - df_monthly_failure_perc['Failure_Rate']
df_monthly_failure_perc['Launches'] = clean_df.groupby('Year_of_Launch').size().values
from plotly.subplots import make_subplots
import plotly.graph_objects as go
df_monthly_failure_perc.head()

,Year_of_Launch,Failure_Rate,Success_Rate,Launches
0,"1,957.00",33.33,66.67,3
1,"1,958.00",72.73,27.27,22
2,"1,959.00",60.00,40.00,20
3,"1,960.00",50.00,50.00,38
4,"1,961.00",38.46,61.54,52


In [167]:
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(
    go.Scatter(x=df_monthly_failure_perc['Year_of_Launch'], 
               y=df_monthly_failure_perc['Failure_Rate'], 
               name='Failure Rate'),
    secondary_y=False
)
fig.add_trace(
    go.Scatter(x=df_monthly_failure_perc['Year_of_Launch'], 
               y=df_monthly_failure_perc['Launches'], 
               name='Total Launches'),
    secondary_y=True
)
fig.update_layout(title_text='Failure Rate vs Total Launches per Year')
fig.update_xaxes(title_text='Year of Launch')
fig.update_yaxes(title_text='Failure Rate (%)', secondary_y=False)
fig.update_yaxes(title_text='Total Launches', secondary_y=True)
fig.update_layout(
    legend=dict(
        x=1.01,        # posizione orizzontale (>1 = fuori dal grafico)
        y=1,           # posizione verticale
        font=dict(size=10)  # testo più piccolo
    )
)
fig.show()


# For Every Year Show which Country was in the Lead in terms of Total Number of Launches up to and including including 2020)

Do the results change if we only look at the number of successful launches? 

In [181]:
# Passo 1: aggrega
df_by_year_country = clean_df.groupby(['Year_of_Launch', 'Country']).size().reset_index(name='Launches')

# Passo 2: trova il leader per ogni anno
df_lead = df_by_year_country.loc[df_by_year_country.groupby('Year_of_Launch')['Launches'].idxmax()]


In [183]:
fig = px.bar(df_lead, 
              x='Year_of_Launch', 
              y='Launches', title='Country Leaders in Space Launches',
              color ='Country',
              color_continuous_scale='Agsunset',
              labels={'Year_of_Launch': 'Year of Launch', 'Rolling_Avg': '12-Month Rolling Average of Launches'}
              )
fig.update_layout(
                xaxis=dict(
                    title='Year of Launch',
                    nticks=20,          # main tick every 20 years
                ),
                yaxis_title='Number of Launches',
                coloraxis_showscale=False                         
                )
fig.show()

In [184]:
# Passo 1: aggrega
df_by_year_country_suc = clean_df[clean_df['Mission_Status'] == 'Success'].groupby(['Year_of_Launch', 'Country']).size().reset_index(name='Launches')

# Passo 2: trova il leader per ogni anno
df_lead_suc = df_by_year_country_suc.loc[df_by_year_country_suc.groupby('Year_of_Launch')['Launches'].idxmax()]


In [185]:
fig = px.bar(df_lead_suc, 
              x='Year_of_Launch', 
              y='Launches', title='Country Leaders in Space Launches',
              color ='Country',
              color_continuous_scale='Agsunset',
              labels={'Year_of_Launch': 'Year of Launch', 'Rolling_Avg': '12-Month Rolling Average of Launches'}
              )
fig.update_layout(
                xaxis=dict(
                    title='Year of Launch',
                    nticks=20,          # main tick every 20 years
                ),
                yaxis_title='Number of Launches',
                coloraxis_showscale=False                         
                )
fig.show()

# Create a Year-on-Year Chart Showing the Organisation Doing the Most Number of Launches

Which organisation was dominant in the 1970s and 1980s? Which organisation was dominant in 2018, 2019 and 2020? 

In [187]:
# Passo 1: aggrega
df_by_year_organisation_suc = clean_df[clean_df['Mission_Status'] == 'Success'].groupby(['Year_of_Launch', 'Organisation']).size().reset_index(name='Launches')

# Passo 2: trova il leader per ogni anno
df_lead_org_suc = df_by_year_organisation_suc.loc[df_by_year_organisation_suc.groupby('Year_of_Launch')['Launches'].idxmax()]


In [192]:
fig = px.bar(df_lead_org_suc, 
              x='Year_of_Launch', 
              y='Launches', title='Organisation Leaders in Space Launches',
              color ='Organisation',
              color_continuous_scale='Agsunset',
              labels={'Year_of_Launch': 'Year of Launch', 'Rolling_Avg': '12-Month Rolling Average of Launches'}
              )
fig.update_layout(
                xaxis=dict(
                    title='Year of Launch',
                    nticks=20,          # main tick every 20 years
                ),
                yaxis_title='Number of Launches',
                coloraxis_showscale=False                         
                )
fig.show()

In [195]:
print(df_lead_org_suc[df_lead_org_suc['Year_of_Launch'] >= 2018])

     Year_of_Launch Organisation  Launches
530        2,018.00         CASC        37
544        2,019.00         CASC        26
558        2,020.00         CASC        17
